# RAG con guías de viaje PDF

Clasificación: **RAG-augmented agents.** Los agentes consultan guías de viaje indexadas en un vectorstore antes de planificar, usando datos reales de las guías como base.

## Qué aporta RAG aquí

Sin RAG, los agentes generan actividades y rutas de memoria (o buscando en web). Con RAG, primero consultan guías de viaje PDF indexadas en ChromaDB y usan esa información como contexto. La búsqueda web complementa si faltan datos.

## Cómo funciona RagTool

`RagTool` de CrewAI indexa documentos (PDF, web, texto) en un vectorstore y los consulta por similitud semántica:

```python
from crewai_tools import RagTool

rag = RagTool()
rag.add(data_type="file", path="input/guia-islandia.pdf")

# El agente lo usa como cualquier otra tool
agente = Agent(..., tools=[rag])
```

El agente puede invocar la tool con una query ("rutas en el sur de Islandia") y recibe los fragmentos más relevantes de las guías indexadas.

In [ ]:
!uv pip install -r requirements.txt --quiet

In [ ]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

## Indexar guías PDF

Coloca PDFs de guías de viaje en `input/` y ejecuta esta celda. Solo necesitas hacerlo una vez.

In [ ]:
import os
from crewai_tools import RagTool

rag = RagTool()

os.makedirs("input", exist_ok=True)

pdfs = [f for f in os.listdir("input") if f.lower().endswith(".pdf")]
if pdfs:
    for i, f in enumerate(pdfs, 1):
        print(f"[{i}/{len(pdfs)}] Indexando: {f}")
        rag.add(data_type="file", path=f"input/{f}")
    print("Guías indexadas.")
else:
    print("No hay PDFs en input/. El agente solo usará búsqueda web.")

## Flow con RAG + Routing

El flow pide datos, clasifica el destino (ciudad/país), y lanza la crew con el `RagTool` asignado al agente de actividades para que consulte las guías indexadas.

In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.flow.flow import Flow, start, listen, router
from crewai_tools import SerperDevTool
from pydantic import BaseModel
from viajes_crew import ViajesCrew
from tools import google_maps_distance


class TripRagState(BaseModel):
    destino: str = ""
    dias: int = 0
    personas: int = 0
    presupuesto: int = 0
    categoria: str = ""
    itinerario: str = ""


class TripPlannerRagFlow(Flow[TripRagState]):

    @start()
    def pedir_datos(self):
        print("\n=== Planificador con RAG ===\n")
        self.state.destino = input("Destino: ")
        self.state.dias = int(input("Días: "))
        self.state.personas = int(input("Personas: "))
        self.state.presupuesto = int(input("Presupuesto (EUR): "))
        return self.state

    @listen(pedir_datos)
    def clasificar(self, state):
        clasificador = Agent(
            role="Clasificador de Alcance de Viaje",
            goal="Determinar si un viaje es a una ciudad o a un pais/region",
            backstory="Distingues viajes urbanos de viajes que implican recorrer varias zonas.",
        )
        task = Task(
            description=f'Clasifica "{state.destino}" como CIUDAD o PAIS_REGION. Responde solo con la palabra.',
            expected_output="Una sola palabra: CIUDAD o PAIS_REGION.",
            agent=clasificador,
        )
        result = Crew(agents=[clasificador], tasks=[task]).kickoff()
        self.state.categoria = result.raw.strip().upper()
        print(f"Destino: {state.destino} -> {self.state.categoria}")

    @router(clasificar)
    def decidir_ruta(self):
        if self.state.categoria == "PAIS_REGION":
            return "con_coche"
        return "sin_coche"

    @listen("con_coche")
    def planificar_con_coche(self):
        crew_instance = ViajesCrew()
        # Añadir RAG al agente de actividades
        actividades = Agent(
            config=crew_instance.agents_config["actividades"],
            tools=[SerperDevTool(), rag],
        )
        itinerario = Agent(
            config=crew_instance.agents_config["itinerario"],
            tools=[google_maps_distance],
        )
        inputs = {
            "destino": self.state.destino,
            "dias": self.state.dias,
            "personas": self.state.personas,
            "presupuesto": self.state.presupuesto,
            "tipo_coche": "alquiler",
        }
        agents = [
            crew_instance.vuelos(), crew_instance.alojamiento(),
            actividades, crew_instance.transporte(),
            crew_instance.coche(), itinerario,
        ]
        tasks = [
            crew_instance.vuelos_task(), crew_instance.alojamiento_task(),
            crew_instance.actividades_task(), crew_instance.transporte_task(),
            crew_instance.coche_task(), crew_instance.itinerario_task(),
        ]
        result = Crew(agents=agents, tasks=tasks, process=Process.sequential, verbose=True).kickoff(inputs=inputs)
        self.state.itinerario = result.raw
        return result.raw

    @listen("sin_coche")
    def planificar_sin_coche(self):
        crew_instance = ViajesCrew()
        # Añadir RAG al agente de actividades
        actividades = Agent(
            config=crew_instance.agents_config["actividades"],
            tools=[SerperDevTool(), rag],
        )
        itinerario = Agent(
            config=crew_instance.agents_config["itinerario"],
            tools=[google_maps_distance],
        )
        inputs = {
            "destino": self.state.destino,
            "dias": self.state.dias,
            "personas": self.state.personas,
            "presupuesto": self.state.presupuesto,
        }
        agents = [
            crew_instance.vuelos(), crew_instance.alojamiento(),
            actividades, crew_instance.transporte(), itinerario,
        ]
        tasks = [
            crew_instance.vuelos_task(), crew_instance.alojamiento_task(),
            crew_instance.actividades_task(), crew_instance.transporte_task(),
            crew_instance.itinerario_task(),
        ]
        result = Crew(agents=agents, tasks=tasks, process=Process.sequential, verbose=True).kickoff(inputs=inputs)
        self.state.itinerario = result.raw
        return result.raw

## Ejecución

In [ ]:
flow = TripPlannerRagFlow()
result = flow.kickoff()
print(result)